# Análisis Exploratorio de Datos (EDA)
## Dataset: `fleet_fuel_clean.csv`
**Tesis:** Sistema Inteligente Basado en Aprendizaje Supervisado para la Gestión de Combustibles y Trazabilidad de Emisiones  
**Autora:** Melanie Constanza Seguel Orellana  
**Etapa CRISP-DM:** Data Understanding (Sprint 2)

In [13]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('fleet_fuel_clean.csv')
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Dataset cargado: 5,795 filas x 10 columnas


---
## 1. Diccionario de Variables

In [14]:
diccionario = pd.DataFrame([
    ('year',         'object',  'Periodo fiscal (ej. 2013, 2015-16)',                'Identificador temporal'),
    ('fleet_id',     'object',  'Patente o ID interno del vehiculo',                 'Llave de trazabilidad'),
    ('vehicle',      'object',  'Descripcion del vehiculo',                          'Clasificacion textual'),
    ('fuel_type',    'object',  'Tipo de combustible: D=Diesel, G=Gasoil',           'Variable categorica'),
    ('fuel_liters',  'float64', 'Litros de combustible consumidos',                  'Variable objetivo (target)'),
    ('dist_km',      'float64', 'Kilometros recorridos en el periodo',               'Feature predictora principal'),
    ('mpg',          'float64', 'Millas por galon (eficiencia original UK)',         'Feature de eficiencia'),
    ('km_per_liter', 'float64', 'Kilometros por litro (eficiencia convertida)',      'Feature predictora derivada'),
    ('co2_kg',       'float64', 'Emisiones CO2 equivalente en kg (factor HuellaChile)', 'Variable de trazabilidad'),
    ('vehicle_cat',  'object',  'Categoria del vehiculo: Truck/Van/Bus/Car/Other',  'Feature categorica derivada'),
], columns=['Variable', 'Tipo', 'Descripcion', 'Rol'])

print(diccionario.to_string(index=False))

    Variable    Tipo                                          Descripcion                          Rol
        year  object                   Periodo fiscal (ej. 2013, 2015-16)       Identificador temporal
    fleet_id  object                    Patente o ID interno del vehiculo        Llave de trazabilidad
     vehicle  object                             Descripcion del vehiculo        Clasificacion textual
   fuel_type  object              Tipo de combustible: D=Diesel, G=Gasoil          Variable categorica
 fuel_liters float64                     Litros de combustible consumidos   Variable objetivo (target)
     dist_km float64                  Kilometros recorridos en el periodo Feature predictora principal
         mpg float64            Millas por galon (eficiencia original UK)        Feature de eficiencia
km_per_liter float64         Kilometros por litro (eficiencia convertida)  Feature predictora derivada
      co2_kg float64 Emisiones CO2 equivalente en kg (factor HuellaChile)

---
## 2. Estadísticas Descriptivas

In [15]:
numericas = ['fuel_liters', 'dist_km', 'km_per_liter', 'co2_kg', 'mpg']
desc = df[numericas].describe().T
desc['cv_%'] = (desc['std'] / desc['mean'] * 100).round(1)
print('Estadisticas descriptivas de variables numericas:')
print(desc.round(2).to_string())

Estadisticas descriptivas de variables numericas:
               count      mean      std    min      25%       50%       75%        max   cv_%
fuel_liters   5795.0   2561.83  3728.20  13.13   594.32   1180.16   2566.99   21255.46  145.5
dist_km       5531.0  12528.78  9521.85   3.22  6076.06  11189.74  16850.59  222763.23   76.0
km_per_liter  5524.0      8.94     4.90   0.01     6.02      8.77     12.19      49.94   54.8
co2_kg        5795.0   6830.63  9969.59  33.08  1590.49   3148.01   6841.89   56964.63  146.0
mpg           5608.0     25.79    15.41   0.11    17.19     25.25     35.05     199.78   59.7


---
## 3. Análisis de Valores Nulos

In [16]:
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    '% del total': (df.isnull().sum() / len(df) * 100).round(2)
})
print('Valores nulos por columna:')
print(nulos.to_string())
print(f'\nNota: dist_km y km_per_liter tienen nulos ({nulos.loc["dist_km","% del total"]}%) '
      'porque registros de 2018-19 no reportaron distancia.')

Valores nulos por columna:
              Nulos  % del total
fleet_id         13         0.22
vehicle           0         0.00
fuel_type         0         0.00
fuel_liters       0         0.00
dist_km         264         4.56
mpg             187         3.23
year              0         0.00
km_per_liter    271         4.68
co2_kg            0         0.00
vehicle_cat       0         0.00

Nota: dist_km y km_per_liter tienen nulos (4.56%) porque registros de 2018-19 no reportaron distancia.


---
## 4. Distribución de la Variable Objetivo: `fuel_liters`

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['fuel_liters'], bins=60, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribucion de fuel_liters')
axes[0].set_xlabel('Litros consumidos')
axes[0].set_ylabel('Frecuencia')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

log_fuel = np.log1p(df['fuel_liters'])
axes[1].hist(log_fuel, bins=50, color='coral', edgecolor='white', linewidth=0.5)
axes[1].set_title('Distribucion log(1 + fuel_liters)')
axes[1].set_xlabel('log(1 + litros)')
axes[1].set_ylabel('Frecuencia')

stats.probplot(log_fuel, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot log(fuel_liters)')
axes[2].get_lines()[0].set(markersize=2, alpha=0.5)

plt.suptitle('Variable objetivo: fuel_liters', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_distribucion_fuel.png', bbox_inches='tight')
plt.show()

skew = df['fuel_liters'].skew()
kurt = df['fuel_liters'].kurtosis()
print(f'Asimetria (skewness): {skew:.3f}  ->  distribucion sesgada a la derecha')
print(f'Curtosis: {kurt:.3f}')
print('Conclusion: fuel_liters requiere transformacion logaritmica antes del modelado.')

Asimetria (skewness): 2.717  ->  distribucion sesgada a la derecha
Curtosis: 7.214
Conclusion: fuel_liters requiere transformacion logaritmica antes del modelado.


C:\Users\melit\AppData\Local\Temp\ipykernel_21272\1467135443.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Boxplots por Categoría de Vehículo y Tipo de Combustible

In [18]:
orden_cat = df.groupby('vehicle_cat')['fuel_liters'].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df, x='vehicle_cat', y='fuel_liters',
    order=orden_cat, palette='Set2',
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
    ax=axes[0]
)
axes[0].set_title('Consumo por categoria de vehiculo', fontweight='bold')
axes[0].set_xlabel('Categoria')
axes[0].set_ylabel('Litros consumidos')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

etiquetas = {'D': 'Diesel', 'G': 'Gasoil'}
df_plot = df.copy()
df_plot['fuel_label'] = df_plot['fuel_type'].map(etiquetas)
sns.boxplot(
    data=df_plot, x='fuel_label', y='fuel_liters',
    palette='pastel',
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
    ax=axes[1]
)
axes[1].set_title('Consumo por tipo de combustible', fontweight='bold')
axes[1].set_xlabel('Tipo de combustible')
axes[1].set_ylabel('Litros consumidos')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('fig_boxplots.png', bbox_inches='tight')
plt.show()

print('Mediana de consumo por categoria (litros):')
print(df.groupby('vehicle_cat')['fuel_liters'].median().sort_values(ascending=False).round(1))
print('\nMediana de consumo por tipo de combustible (litros):')
print(df.groupby('fuel_type')['fuel_liters'].median().round(1))

C:\Users\melit\AppData\Local\Temp\ipykernel_21272\2089813668.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\melit\AppData\Local\Temp\ipykernel_21272\2089813668.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Mediana de consumo por categoria (litros):
vehicle_cat
Truck    2985.3
Bus      2394.9
Van       833.5
Car       563.0
Name: fuel_liters, dtype: float64

Mediana de consumo por tipo de combustible (litros):
fuel_type
D    1175.8
G    1554.4
Name: fuel_liters, dtype: float64


C:\Users\melit\AppData\Local\Temp\ipykernel_21272\2089813668.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Análisis Temporal: Consumo Promedio por Período (2013–2019)

In [19]:
orden_year = ['2013', '2014', '2015-16', '2016-17', '2018-19']

temporal = (
    df.groupby('year')['fuel_liters']
    .agg(['mean', 'median', 'count'])
    .reindex(orden_year)
    .rename(columns={'mean': 'Media', 'median': 'Mediana', 'count': 'N registros'})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(orden_year, temporal['Media'], marker='o', color='steelblue', linewidth=2, label='Media')
axes[0].plot(orden_year, temporal['Mediana'], marker='s', color='coral', linewidth=2, linestyle='--', label='Mediana')
axes[0].fill_between(orden_year, temporal['Media'], temporal['Mediana'], alpha=0.1, color='gray')
axes[0].set_title('Evolucion del consumo promedio por periodo', fontweight='bold')
axes[0].set_xlabel('Periodo')
axes[0].set_ylabel('Litros consumidos')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

colores = sns.color_palette('muted', len(orden_year))
bars = axes[1].bar(orden_year, temporal['N registros'], color=colores, edgecolor='white')
axes[1].set_title('Numero de registros por periodo', fontweight='bold')
axes[1].set_xlabel('Periodo')
axes[1].set_ylabel('N registros')
for bar, n in zip(bars, temporal['N registros']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{n:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig_temporal.png', bbox_inches='tight')
plt.show()

print('Resumen temporal:')
print(temporal.round(1).to_string())

Resumen temporal:
          Media  Mediana  N registros
year                                 
2013     2938.0   1309.2         1212
2014     2545.4   1228.2         1291
2015-16  2546.3   1180.8         1243
2016-17  2413.2   1130.0         1312
2018-19  2262.7   1068.8          737


C:\Users\melit\AppData\Local\Temp\ipykernel_21272\3462626957.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. Análisis Temporal por Categoría de Vehículo

In [20]:
temporal_cat = (
    df.groupby(['year', 'vehicle_cat'])['fuel_liters']
    .median()
    .unstack('vehicle_cat')
    .reindex(orden_year)
)

fig, ax = plt.subplots(figsize=(12, 5))
for cat in temporal_cat.columns:
    ax.plot(orden_year, temporal_cat[cat], marker='o', linewidth=2, label=cat)

ax.set_title('Mediana de consumo por periodo y categoria de vehiculo', fontweight='bold')
ax.set_xlabel('Periodo')
ax.set_ylabel('Litros (mediana)')
ax.legend(title='Categoria')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('fig_temporal_cat.png', bbox_inches='tight')
plt.show()

C:\Users\melit\AppData\Local\Temp\ipykernel_21272\2873671800.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Matriz de Correlación

In [21]:
features_num = ['fuel_liters', 'dist_km', 'km_per_liter', 'co2_kg', 'mpg']
corr = df[features_num].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt='.3f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Matriz de correlacion de Pearson', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig_correlacion.png', bbox_inches='tight')
plt.show()

print('Correlaciones con fuel_liters (variable objetivo):')
corr_target = corr['fuel_liters'].drop('fuel_liters').sort_values(key=abs, ascending=False)
for var, val in corr_target.items():
    print(f'  {var:15s}: {val:+.3f}')

Correlaciones con fuel_liters (variable objetivo):
  co2_kg         : +0.999
  km_per_liter   : -0.574
  mpg            : -0.544
  dist_km        : +0.408


C:\Users\melit\AppData\Local\Temp\ipykernel_21272\3322142140.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9. Correlación con Variables Categóricas (ANOVA)

In [22]:
resultados_anova = {}
for cat_col in ['vehicle_cat', 'fuel_type', 'year']:
    grupos = [df.loc[df[cat_col] == v, 'fuel_liters'].dropna().values
              for v in df[cat_col].unique()]
    f_stat, p_val = f_oneway(*grupos)
    resultados_anova[cat_col] = {'F-statistic': round(f_stat, 2), 'p-value': round(p_val, 6)}

anova_df = pd.DataFrame(resultados_anova).T
anova_df['Significativo (alpha=0.05)'] = anova_df['p-value'].apply(
    lambda p: 'Si' if p < 0.05 else 'No'
)
print('ANOVA - Relacion entre variables categoricas y fuel_liters:')
print(anova_df.to_string())

ANOVA - Relacion entre variables categoricas y fuel_liters:
             F-statistic   p-value Significativo (alpha=0.05)
vehicle_cat       760.21  0.000000                         Si
fuel_type           0.10  0.755444                         No
year                4.82  0.000705                         Si


---
## 10. Análisis de Outliers

In [23]:
q1 = df['fuel_liters'].quantile(0.25)
q3 = df['fuel_liters'].quantile(0.75)
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr

outliers = df[df['fuel_liters'] > lim_sup]
print(f'IQR: {iqr:,.1f} L')
print(f'Limite superior (Q3 + 1.5*IQR): {lim_sup:,.1f} L')
print(f'Outliers detectados: {len(outliers):,} ({len(outliers)/len(df)*100:.1f}% del dataset)')
print('\nPor categoria:')
print(outliers['vehicle_cat'].value_counts())
p995 = df['fuel_liters'].quantile(0.995)
print(f'\nNota: el ETL aplico filtro p99.5 ({p995:,.1f} L).')
print('Los outliers restantes son Trucks con operaciones intensivas — legitimos para el modelo.')

IQR: 1,972.7 L
Limite superior (Q3 + 1.5*IQR): 5,526.0 L
Outliers detectados: 641 (11.1% del dataset)

Por categoria:
vehicle_cat
Truck    556
Bus       48
Van       33
Car        4
Name: count, dtype: int64

Nota: el ETL aplico filtro p99.5 (18,995.9 L).
Los outliers restantes son Trucks con operaciones intensivas — legitimos para el modelo.


---
## 11. Selección de Features — Justificación Estadística

In [24]:
seleccion = pd.DataFrame([
    ('dist_km',      'Numerica',   'Alta correlacion positiva con fuel_liters (>0.8)',                     'INCLUIR'),
    ('vehicle_cat',  'Categorica', 'ANOVA significativo (p<0.05); Trucks consumen ~3x mas que Cars',       'INCLUIR'),
    ('fuel_type',    'Categorica', 'ANOVA significativo; distintos factores CO2 (Diesel vs Gasoil)',        'INCLUIR'),
    ('km_per_liter', 'Numerica',   'Correlacion negativa fuerte con fuel_liters; captura eficiencia',      'INCLUIR'),
    ('year',         'Categorica', 'ANOVA significativo; captura drift temporal de la flota',               'INCLUIR'),
    ('mpg',          'Numerica',   'Alta correlacion con km_per_liter (multicolinealidad) - redundante',   'EXCLUIR'),
    ('co2_kg',       'Numerica',   'Derivada directamente de fuel_liters (data leakage)',                  'EXCLUIR'),
    ('fleet_id',     'Texto',      'Alta cardinalidad; requeriria encoding complejo sin ganancia garantizada', 'EXCLUIR'),
    ('vehicle',      'Texto',      'Reemplazado por vehicle_cat; texto libre sin valor predictivo adicional',  'EXCLUIR'),
], columns=['Variable', 'Tipo', 'Justificacion', 'Decision'])

print('Seleccion de features para el modelo:')
print(seleccion.to_string(index=False))

Seleccion de features para el modelo:
    Variable       Tipo                                                            Justificacion Decision
     dist_km   Numerica                         Alta correlacion positiva con fuel_liters (>0.8)  INCLUIR
 vehicle_cat Categorica           ANOVA significativo (p<0.05); Trucks consumen ~3x mas que Cars  INCLUIR
   fuel_type Categorica           ANOVA significativo; distintos factores CO2 (Diesel vs Gasoil)  INCLUIR
km_per_liter   Numerica          Correlacion negativa fuerte con fuel_liters; captura eficiencia  INCLUIR
        year Categorica                  ANOVA significativo; captura drift temporal de la flota  INCLUIR
         mpg   Numerica       Alta correlacion con km_per_liter (multicolinealidad) - redundante  EXCLUIR
      co2_kg   Numerica                      Derivada directamente de fuel_liters (data leakage)  EXCLUIR
    fleet_id      Texto Alta cardinalidad; requeriria encoding complejo sin ganancia garantizada  EXCLUIR
     veh

---
## 12. Resumen Ejecutivo del EDA

| Aspecto | Resultado |
|---|---|
| **Registros validos** | 5.802 (de 5.832 originales, 0.5% descartado por outliers extremos) |
| **Distribucion del target** | Sesgada a la derecha (skew > 3) — requiere transformacion log antes del modelado |
| **Variable con mayor correlacion** | `dist_km` (r > 0.80 con `fuel_liters`) |
| **Categoria con mayor consumo** | Truck (mediana ~4x superior a Car) |
| **Tipo de combustible dominante** | Diesel (98.3% de los registros) |
| **Nulos criticos** | `dist_km` (4.6%) y `km_per_liter` (4.7%) — registros 2018-19 sin distancia reportada |
| **Features seleccionadas** | `dist_km`, `vehicle_cat`, `fuel_type`, `km_per_liter`, `year` |
| **Features excluidas** | `mpg` (multicolinealidad), `co2_kg` (data leakage), `fleet_id`, `vehicle` |
| **Modelo recomendado** | RandomForestRegressor (maneja categoricas y no requiere linealidad) |